In [10]:
# ============================================
# TITANIC SURVIVAL PREDICTION
# ============================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, GridSearchCV

# Load data
train_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/titanic_train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/titanic_test.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train shape: (712, 10)
Test shape: (179, 10)


In [11]:
# ============================================
# STEP 1 - EXPLORE THE DATA
# ============================================

print("First 5 rows:")
print(train_df.head())

print("\nMissing values in train:")
print(train_df.isnull().sum())

print("\nUnique values in sex column:")
print(train_df['sex'].unique())

First 5 rows:
   Survived  pclass     sex   age  sibsp  parch      fare embarked  \
0         1       3    male   NaN      0      0   56.4958        S   
1         0       2       M   NaN      0      0    0.0000        S   
2         0       1    Male   NaN      0      0  221.7792        S   
3         1       3  female  18.0      0      1       NaN        S   
4         1       2  female  31.0      1      1   26.2500        S   

   ticket_type  has_cabin_id  
0     3.054184             1  
1     1.933670             0  
2     0.984326             1  
3     2.837520             0  
4     2.013784             0  

Missing values in train:
Survived          0
pclass            0
sex               0
age             216
sibsp             0
parch             0
fare             89
embarked         86
ticket_type       0
has_cabin_id      0
dtype: int64

Unique values in sex column:
['male' 'M' 'Male' 'female' 'Female' 'F']


In [12]:
# ============================================
# STEP 2 - CLEAN SEX COLUMN
# Problem: 6 variations (male, M, Male, female, F, Female)
# Fix: Standardize to lowercase then map to 0/1
# ============================================

train_df['sex'] = train_df['sex'].str.lower().str.strip()
test_df['sex'] = test_df['sex'].str.lower().str.strip()

sex_map = {'male': 0, 'm': 0, 'female': 1, 'f': 1}
train_df['sex'] = train_df['sex'].map(sex_map)
test_df['sex'] = test_df['sex'].map(sex_map)

print("Sex unique values after cleaning:", train_df['sex'].unique())

Sex unique values after cleaning: [0 1]


In [13]:
# ============================================
# STEP 3 - FIX MISSING AGE
# Problem: 216 missing values in train
# Fix: Fill with median age from training data
# ============================================

median_age = train_df['age'].median()
print("Median age used for filling:", median_age)

train_df['age'] = train_df['age'].fillna(median_age)
test_df['age'] = test_df['age'].fillna(median_age)

print("Missing age in train:", train_df['age'].isnull().sum())
print("Missing age in test:", test_df['age'].isnull().sum())

Median age used for filling: 29.0
Missing age in train: 0
Missing age in test: 0


In [14]:
# ============================================
# STEP 4 - FIX MISSING FARE
# Problem: 89 missing values in train
# Fix: Fill with class-based median (smarter than overall median)
# Reason: 1st class fare is very different from 3rd class fare
# ============================================

print("Median fare by class:")
print(train_df.groupby('pclass')['fare'].median())

train_df['fare'] = train_df.groupby('pclass')['fare'].transform(
    lambda x: x.fillna(x.median()))
test_df['fare'] = test_df.groupby('pclass')['fare'].transform(
    lambda x: x.fillna(x.median()))

print("\nMissing fare in train:", train_df['fare'].isnull().sum())
print("Missing fare in test:", test_df['fare'].isnull().sum())

Median fare by class:
pclass
1    56.9292
2    15.0458
3     8.0500
Name: fare, dtype: float64

Missing fare in train: 0
Missing fare in test: 0


In [15]:
# ============================================
# STEP 5 - FIX MISSING EMBARKED
# Problem: 86 missing values, text column
# Fix: Fill with most common port (S), convert to numbers
# ============================================

print("Port counts:")
print(train_df['embarked'].value_counts())

train_df['embarked'] = train_df['embarked'].fillna('S')
test_df['embarked'] = test_df['embarked'].fillna('S')

embarked_map = {'S': 0, 'C': 1, 'Q': 2}
train_df['embarked'] = train_df['embarked'].map(embarked_map)
test_df['embarked'] = test_df['embarked'].map(embarked_map)

print("Missing embarked in train:", train_df['embarked'].isnull().sum())

Port counts:
embarked
S    455
C    126
Q     45
Name: count, dtype: int64
Missing embarked in train: 0


In [16]:
# ============================================
# STEP 6 - FEATURE ENGINEERING
# Creating new meaningful features from existing columns
# ============================================

# Family size - traveling with family affected survival
train_df['family_size'] = train_df['sibsp'] + train_df['parch'] + 1
test_df['family_size'] = test_df['sibsp'] + test_df['parch'] + 1

# Is alone flag
train_df['is_alone'] = (train_df['family_size'] == 1).astype(int)
test_df['is_alone'] = (test_df['family_size'] == 1).astype(int)

print("Sample of new features:")
print(train_df[['sibsp', 'parch', 'family_size', 'is_alone']].head())

Sample of new features:
   sibsp  parch  family_size  is_alone
0      0      0            1         1
1      0      0            1         1
2      0      0            1         1
3      0      1            2         0
4      1      1            3         0


In [8]:
# ============================================
# STEP 7 - TRAIN AND TUNE MODEL
# Using GridSearchCV to find best settings
# ============================================

features = ['sex', 'ticket_type', 'fare', 'age',
            'pclass', 'family_size', 'embarked']

X_train = train_df[features]
y_train = train_df['Survived']
X_test = test_df[features]
y_test = test_df['Survived']

# GridSearchCV - tries 135 combinations automatically
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 5, 6, 7, 8],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5,
    scoring='accuracy',
    n_jobs=-1, verbose=1
)

grid_search.fit(X_train, y_train)
print("Best settings:", grid_search.best_params_)
print(f"Best CV Accuracy: {grid_search.best_score_*100:.2f}%")

Fitting 5 folds for each of 135 candidates, totalling 675 fits
Best settings: {'max_depth': 7, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Best CV Accuracy: 83.16%


In [9]:
# ============================================
# STEP 8 - FINAL RESULTS
# ============================================

best_model = grid_search.best_estimator_
test_preds = best_model.predict(X_test)

print("=" * 40)
print("RESULTS SUMMARY")
print("=" * 40)
print(f"Baseline Accuracy:     ~72%")
print(f"Our CV Accuracy:        {grid_search.best_score_*100:.2f}%")
print(f"Our Test Accuracy:      {accuracy_score(y_test, test_preds)*100:.2f}%")
print("=" * 40)

# Feature importance
importance = pd.DataFrame({
    'feature': features,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(importance)

# Save submission
submission = pd.DataFrame({
    'Actual': y_test,
    'Predicted': test_preds
})
submission.to_csv('titanic_submission.csv', index=False)
print("\nSubmission file saved!")

RESULTS SUMMARY
Baseline Accuracy:     ~72%
Our CV Accuracy:        83.16%
Our Test Accuracy:      77.65%

Feature Importance:
       feature  importance
0          sex    0.379101
1  ticket_type    0.181383
2         fare    0.163410
3          age    0.120334
4       pclass    0.064013
5  family_size    0.058665
6     embarked    0.033095

Submission file saved!
